# ENSEMBLE QUANTUM — K-Fold Cross-Validation + Ensemble
### All 13 Quantum Models | Conjunctiva Image → Hb Estimation

**Workflow (strict — test set never touched during training):**

```
All Data
  │
  ├── Test Set (held-out, fixed) ──────────────────────────────► Final eval only
  │
  └── Train+Val Set
        │
        └── K-Fold Cross-Validation (per model)
              ├── Fold 1: train on K-1, predict OOF on fold 1
              ├── Fold 2: train on K-1, predict OOF on fold 2
              ├── ...
              └── Fold K: → collect OOF predictions for ALL models
                              │
                              ├── Weighted Average Ensemble  (by CV score)
                              ├── Stacking Ensemble          (meta-learner on OOF)
                              └── Best-Single-Model
                                        │
                                        └── Retrain on full Train+Val → Predict Test
```

| # | Model | Type | Publication |
|---|-------|------|-------------|
| 1 | QSVM | SVM | Nature 567 (2019) — IBM |
| 2 | VQC | Gradient | arXiv 1802.06002 (2018) — Google |
| 3 | QCNN | Gradient | Nature Physics 15 (2019) — Harvard |
| 4 | TorchQuantum VQA | Gradient | Nat. Rev. Physics 3 (2021) — MIT |
| 5 | Data Re-Uploading | Gradient | Quantum 4, 226 (2020) |
| 6 | Quantum Transfer Learning | Gradient | Quantum 4, 340 (2020) — Xanadu |
| 7 | Quantum Kernel | SVM | PRX Quantum (2021) — Xanadu |
| 8 | Quantum Random Forest | SVM | Springer QMI (2023) |
| 9 | QBoost | SVM | ACML 2012 — D-Wave |
| 10 | CV-QNN | Gradient | Phys Rev Research 1 (2019) — Xanadu |
| B1 | QSVM Medical | SVM | Bonus |
| B2 | QCNN Phases | Gradient | Bonus |
| B3 | Meta-Learning QNN | Gradient | Bonus |


## Section 1 — CONFIGURATION  (Edit only this cell)

In [ ]:
# ==============================================================
#         ALL VARIABLES — EDIT ONLY THIS BLOCK
# ==============================================================

TASK = "regression"   # "regression" | "classification"

IMAGE_DIR  = "/kaggle/input/your-dataset/images"
EXCEL_PATH = "/kaggle/input/your-dataset/data.xlsx"
IMAGE_COL  = "Patient_ID"
HB_COL     = "Hemoglobin"

HB_THRESH     = 12.0
HB_FILTER_MIN = 5.0
HB_FILTER_MAX = 18.0

# ── K-Fold settings ───────────────────────────────────────────
K_FOLDS    = 5
TEST_SPLIT = 0.15
SEED       = 42

# ── Quantum circuit ───────────────────────────────────────────
N_QUBITS = 4
N_LAYERS = 2

# ── Backbone ──────────────────────────────────────────────────
FREEZE_BACKBONE = True

# ── Training — increased for better convergence ───────────────
EPOCHS_PER_FOLD = 15     # was 8  — more training per CV fold
EPOCHS_FINAL    = 25     # was 15 — more for final retrain
BATCH_SIZE      = 16
LR              = 1e-3
EARLY_STOP      = 8      # was 3  — less aggressive stopping

# ── Regression config ─────────────────────────────────────────
REG_CONFIG = {
    "normalize_hb": True,
    "hb_min": 4.0, "hb_max": 20.0,
    "loss_fn": "huber", "huber_delta": 1.0,
}

# ── Classification config ─────────────────────────────────────
CLS_CONFIG = {
    "use_focal_loss": True, "focal_gamma": 2.0,
    "use_class_weights": True, "threshold": 0.5,
}

# ── Models to include ─────────────────────────────────────────
RUN = {
    "QSVM"                   : True,
    "VQC"                    : True,
    "QCNN"                   : True,
    "TorchQuantum_VQA"       : True,
    "DataReUploading"        : True,
    "QuantumTransferLearning" : True,
    "QuantumKernel"          : True,
    "QuantumRandForest"      : True,
    "QBoost"                 : True,
    "CVQNN"                  : True,
    "QSVM_Medical"           : True,
    "QCNN_Phases"            : True,
    "MetaLearning_QNN"       : True,
}

# ── Ensemble methods ──────────────────────────────────────────
RUN_ENSEMBLE = {
    "simple_avg"       : True,   # plain mean
    "weighted_avg"     : True,   # weight ∝ CV score
    "rank_avg"         : True,   # rank-based average (outlier-robust)
    "stacking_ridge"   : True,   # Ridge / LogReg meta-learner
    "stacking_gbm"     : True,   # GBM meta-learner
    "voting"           : True,   # median (reg) / majority vote (cls)
    "best_single"      : True,   # best individual model (reference)
    # ── NEW ensemble methods ──────────────────────────────────
    "top_k"            : True,   # mean of top-K models by CV score
    "trimmed_mean"     : True,   # mean after removing top/bottom outlier predictions
    "neural_stacker"   : True,   # small MLP meta-learner on OOF matrix
    "blending"         : True,   # hold-out blending (10% of TV set as blend-val)
    "bayesian_avg"     : True,   # weights ∝ exp(-CV_score/T) — softmax over scores
}

# ── Top-K setting ─────────────────────────────────────────────
TOP_K = 5    # number of best models to include in top_k ensemble


## Step 0 — Clone Repo  *(run once on Kaggle / Colab)*
> On local: skip this cell if you already have the repo checked out.

In [ ]:
import os, sys, subprocess

REPO_URL  = "https://github.com/junaidmaqbool/QuantumHb.git"
CLONE_DIR = "/kaggle/working/QuantumHb"

if not os.path.isdir(CLONE_DIR):
    print("Cloning QuantumHb ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    print(f"✓ Cloned to {CLONE_DIR}")
else:
    print("✓ Repo already present — pulling latest ...")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--rebase"], check=True)

REPO_ROOT = CLONE_DIR
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"TASK            : {TASK}")
print(f"K_FOLDS         : {K_FOLDS}")
print(f"ENSEMBLE_METHOD : {ENSEMBLE_METHOD}")
print(f"N_QUBITS        : {N_QUBITS}")
print(f"Models ON       : {[k for k,v in RUN.items() if v]}")


## Section 2 — Install Dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "pennylane", "pennylane-qiskit",
    "qiskit", "qiskit-machine-learning",
    "torchvision", "timm",
    "pandas", "openpyxl",
    "scikit-learn", "matplotlib",
    "tqdm", "einops", "scipy",
]
for pkg in PKGS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"  {'OK  ' if r.returncode==0 else 'WARN'} {pkg}")
print("Dependencies done.")


## Section 3 — Imports

In [ ]:
import os, sys, math, time, warnings, traceback, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import (train_test_split,
    KFold, StratifiedKFold)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score,
    mean_absolute_error, f1_score, mean_squared_error)
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
np.random.seed(SEED)
# ── CUDA sanity check (guards against compute-capability mismatches) ──
DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        _t = torch.zeros(1, device="cuda")
        _ = _t + _t          # trigger an actual kernel — catches arch mismatches
        del _t
        DEVICE = "cuda"
        print(f"  GPU : {torch.cuda.get_device_name(0)}")
    except Exception as _cuda_err:
        print(f"  ⚠ CUDA available but kernel mismatch → falling back to CPU")
        print(f"    ({_cuda_err})")
        torch.cuda.empty_cache()
print(f"Device  : {DEVICE}   PyTorch: {torch.__version__}")
print(f"Task    : {TASK}   K_FOLDS: {K_FOLDS}   Ensemble: {ENSEMBLE_METHOD}")


---
## Section 3b — Dataset, Test Split & Feature Extraction

**Important:** the test set is isolated first, before any K-fold split,
and is only used at the very end for final ensemble evaluation.


In [ ]:
_IMG_EXTS = [".jpg",".jpeg",".png",".bmp",".tif",".tiff",""]

def _find_img(pid, img_dir):
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(pid) + ext)
        if os.path.exists(p): return p
    return None

TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
TRANSFORM_EVAL = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class HbDataset(Dataset):
    def __init__(self, records, img_dir, transform):
        self.records = records; self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        pid, hb = self.records[idx]
        path = _find_img(pid, self.img_dir)
        img  = Image.open(path).convert("RGB") if path else \
               Image.fromarray(np.zeros((224,224,3), dtype=np.uint8))
        img  = self.transform(img)
        if TASK == "classification":
            label = torch.tensor(1.0 if hb >= HB_THRESH else 0.0)
        else:
            lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
            label  = torch.tensor((hb-lo)/(hi-lo), dtype=torch.float32)
        return img, label

# ── Load & filter ─────────────────────────────────────────────
print("Loading dataset...")
df = pd.read_excel(EXCEL_PATH)
df = df[[IMAGE_COL, HB_COL]].dropna()
df[HB_COL] = pd.to_numeric(df[HB_COL], errors="coerce")  # force numeric; bad strings → NaN
df = df.dropna(subset=[HB_COL])                           # drop rows that failed conversion
df = df[(df[HB_COL]>=HB_FILTER_MIN)&(df[HB_COL]<=HB_FILTER_MAX)].reset_index(drop=True)
records = list(zip(df[IMAGE_COL].astype(str), df[HB_COL].astype(float)))
hb_vals = df[HB_COL].astype(float).values
print(f"  Total samples : {len(records)}")
print(f"  Hb range      : {hb_vals.min():.1f} – {hb_vals.max():.1f} g/dL")

# ── Hold-out test set (fixed, isolated first) ─────────────────
all_idx = np.arange(len(records))
if TASK == "classification":
    labels_strat = (hb_vals >= HB_THRESH).astype(int)
    tv_idx, te_idx = train_test_split(all_idx, test_size=TEST_SPLIT,
                                       random_state=SEED,
                                       stratify=labels_strat)
else:
    tv_idx, te_idx = train_test_split(all_idx, test_size=TEST_SPLIT,
                                       random_state=SEED)

tv_records = [records[i] for i in tv_idx]   # train+val (used in K-fold)
te_records = [records[i] for i in te_idx]   # held-out test set
tv_hb      = hb_vals[tv_idx]
te_hb      = hb_vals[te_idx]

print(f"  Train+Val : {len(tv_records)}  |  Test (held-out) : {len(te_records)}")

# Full datasets (for loader construction inside each fold)
full_dataset = HbDataset(records, IMAGE_DIR, TRANSFORM_EVAL)
test_dataset = HbDataset(te_records, IMAGE_DIR, TRANSFORM_EVAL)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

print("Dataset & test split ready. ✅")


## Section 3c — ResNet-18 Backbone & Feature Caching

In [ ]:
# ── Shared ResNet-18 backbone ─────────────────────────────────
_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
_resnet.fc = nn.Identity()
_resnet = _resnet.to(DEVICE)
if FREEZE_BACKBONE:
    for p in _resnet.parameters(): p.requires_grad_(False)
_resnet.eval()
print(f"ResNet-18 ready  (frozen={FREEZE_BACKBONE})")

# ── Extract 512-dim features for the entire dataset once ──────
print("Pre-extracting ResNet features for all samples (once, shared)...")
_all_ds   = HbDataset(records, IMAGE_DIR, TRANSFORM_EVAL)
_all_ld   = DataLoader(_all_ds, batch_size=32, shuffle=False, num_workers=2)
ALL_FEATS = []   # (N, 512) numpy array
ALL_LABELS= []   # (N,)     numpy array — normalised hb or binary class

with torch.no_grad():
    for imgs, labels in tqdm(_all_ld, desc="ResNet", leave=False):
        ALL_FEATS.append(_resnet(imgs.to(DEVICE)).cpu().numpy())
        ALL_LABELS.append(labels.numpy())

ALL_FEATS  = np.concatenate(ALL_FEATS,  axis=0)   # (N, 512)
ALL_LABELS = np.concatenate(ALL_LABELS, axis=0)   # (N,)

# Split into train+val and test using the same indices
TV_FEATS   = ALL_FEATS[tv_idx]    # (Ntv, 512)
TV_LABELS  = ALL_LABELS[tv_idx]
TE_FEATS   = ALL_FEATS[te_idx]    # (Nte, 512)
TE_LABELS  = ALL_LABELS[te_idx]

# PCA → N_QUBITS (shared pipeline fitted on full train+val)
_sc  = StandardScaler()
_pca = PCA(n_components=N_QUBITS, random_state=SEED)
TV_FEATS_PCA = _pca.fit_transform(_sc.fit_transform(TV_FEATS))
TE_FEATS_PCA = _pca.transform(_sc.transform(TE_FEATS))
print(f"PCA variance retained: {_pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"TV_FEATS_PCA: {TV_FEATS_PCA.shape}  TE_FEATS_PCA: {TE_FEATS_PCA.shape}")
print("Feature extraction complete. ✅")


---
## Section 4 — K-Fold Engine, Ensemble Utilities & Metrics


In [ ]:
# ── Global OOF & test prediction stores ──────────────────────
#   OOF_PREDS  : dict  model_name → np.array (N_tv,)   OOF predictions
#   TEST_PREDS : dict  model_name → np.array (N_te,)   test predictions
#   CV_SCORES  : dict  model_name → float              mean CV metric
OOF_PREDS  = {}
TEST_PREDS = {}
CV_SCORES  = {}
MODEL_RESULTS = []    # per-fold detailed records

# ── Metrics ───────────────────────────────────────────────────
def _denorm_hb(v):
    lo,hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return np.array(v)*(hi-lo)+lo

def _reg_metrics(y_true_n, y_pred_n):
    yt = _denorm_hb(y_true_n); yp = _denorm_hb(y_pred_n)
    mae  = mean_absolute_error(yt, yp)
    rmse = mean_squared_error(yt, yp)**0.5
    return mae, rmse

def _cls_metrics(y_true, y_prob):
    yp = (np.array(y_prob)>=CLS_CONFIG["threshold"]).astype(int)
    acc = accuracy_score(y_true, yp)
    f1  = f1_score(y_true, yp, zero_division=0)
    try:    auc = roc_auc_score(y_true, y_prob)
    except: auc = float("nan")
    return acc, f1, auc

def _score(y_true, y_pred):
    """Primary CV score: lower = better for reg (MAE), higher = better for cls (AUC)."""
    if TASK=="regression":
        mae,_ = _reg_metrics(y_true, y_pred); return mae
    else:
        _,_,auc = _cls_metrics(y_true, y_pred); return auc

# ── Loss functions ────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy(logits, targets, reduction="none")
        return (((1-torch.exp(-bce))**self.gamma)*bce).mean()

def _loss_fn():
    if TASK=="classification":
        return FocalLoss(CLS_CONFIG["focal_gamma"]) if CLS_CONFIG["use_focal_loss"] \
               else nn.BCELoss()
    fn = REG_CONFIG["loss_fn"]
    if fn=="huber": return nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
    if fn=="mae":   return nn.L1Loss()
    return nn.MSELoss()

# ── QuantumNet wrapper ────────────────────────────────────────
class QuantumNet(nn.Module):
    """Frozen ResNet-18 + Linear reducer + quantum model."""
    def __init__(self, qmodel):
        super().__init__()
        self.backbone = _resnet
        self.reducer  = nn.Linear(512, N_QUBITS)
        self.bn       = nn.BatchNorm1d(N_QUBITS)
        self.qmodel   = qmodel
    def forward(self, imgs):
        with torch.set_grad_enabled(not FREEZE_BACKBONE):
            f = self.backbone(imgs)
        x = torch.tanh(self.bn(self.reducer(f))) * math.pi
        return self.qmodel(x)

# ── FeatureDataset (for gradient models — wraps pre-extracted features) ──
class FeatDataset(Dataset):
    """Dataset of pre-extracted (512-dim) features + labels."""
    def __init__(self, feats, labels):
        self.X = torch.tensor(feats,   dtype=torch.float32)
        self.y = torch.tensor(labels,  dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

# ── Thin gradient model (feature input, no ResNet) ────────────
class ThinQuantumNet(nn.Module):
    """Takes pre-extracted 512-dim features directly."""
    def __init__(self, qmodel):
        super().__init__()
        self.reducer = nn.Linear(512, N_QUBITS)
        self.bn      = nn.BatchNorm1d(N_QUBITS)
        self.qmodel  = qmodel
    def forward(self, x):
        x = torch.tanh(self.bn(self.reducer(x))) * math.pi
        return self.qmodel(x)

# ── One gradient fold ─────────────────────────────────────────
def _train_gradient_fold(net, tr_feats, tr_labels, vl_feats, vl_labels,
                          n_epochs, desc=""):
    """Train ThinQuantumNet for one fold. Returns val predictions."""
    net = net.to(DEVICE)
    tr_dl = DataLoader(FeatDataset(tr_feats, tr_labels),
                       batch_size=BATCH_SIZE, shuffle=True)
    vl_dl = DataLoader(FeatDataset(vl_feats, vl_labels),
                       batch_size=BATCH_SIZE, shuffle=False)
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, net.parameters()), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    loss_fn = _loss_fn()
    best_val_loss = float("inf"); patience = 0
    best_state = None

    for epoch in range(n_epochs):
        net.train()
        for xb, yb in tr_dl:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        sched.step()

        # val loss for early stopping
        net.eval(); vl_loss = 0.0
        with torch.no_grad():
            for xb,yb in vl_dl:
                xb,yb = xb.to(DEVICE), yb.to(DEVICE)
                vl_loss += loss_fn(net(xb), yb).item()
        vl_loss /= max(1, len(vl_dl))
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_state    = copy.deepcopy(net.state_dict())
            patience = 0
        else:
            patience += 1
        if EARLY_STOP > 0 and patience >= EARLY_STOP: break

    # Predict on val fold
    if best_state: net.load_state_dict(best_state)
    net.eval()
    preds = []
    with torch.no_grad():
        for xb,_ in vl_dl:
            preds.extend(net(xb.to(DEVICE)).cpu().numpy())
    return net, np.array(preds)

def _predict_gradient(net, feats):
    """Inference with ThinQuantumNet."""
    ds = FeatDataset(feats, np.zeros(len(feats)))
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    net.eval(); preds = []
    with torch.no_grad():
        for xb,_ in dl:
            preds.extend(net(xb.to(DEVICE)).cpu().numpy())
    return np.array(preds)

def _add(folder):
    p = os.path.join(REPO_ROOT, folder)
    if p not in sys.path: sys.path.insert(0, p)

def _load_model(folder, task=TASK):
    import importlib.util as _ilu
    spec = _ilu.spec_from_file_location(
        f"_{folder}", os.path.join(REPO_ROOT, folder, "eye_hb_model.py"))
    mod = _ilu.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod

# ── K-Fold splitter ───────────────────────────────────────────
def _get_kfold():
    if TASK=="classification":
        tv_cls = (tv_hb >= HB_THRESH).astype(int)
        return StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED), tv_cls
    return KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED), tv_hb

print("Utilities ready. ✅")


---
## Section 5 — Per-Model K-Fold Runners

Two runners are defined:
- `run_kfold_gradient(name, folder)` — for PennyLane / TorchQuantum models
- `run_kfold_svm(name, folder)` — for Qiskit / sklearn-compatible models

Each runner:
1. Splits `TV_FEATS_PCA` into K folds
2. Trains on K-1 folds, collects OOF predictions on held-out fold
3. Reports per-fold metric + mean ± std
4. Retrains on the **full** Train+Val set (all K folds combined)
5. Predicts on the **held-out test set**


In [ ]:
def run_kfold_gradient(name, folder):
    """K-fold CV for gradient-based quantum models."""
    print(f"\n{'='*62}")
    print(f"  [GRADIENT] {name}")
    print(f"{'='*62}")
    t0 = time.time()
    kfold, strat_labels = _get_kfold()

    oof_preds  = np.zeros(len(TV_FEATS_PCA))
    fold_scores = []

    try:
        _add(folder)
        for fold, (tr_idx, vl_idx) in enumerate(
                kfold.split(TV_FEATS_PCA, strat_labels), 1):

            print(f"  ── Fold {fold}/{K_FOLDS}  "
                  f"(train={len(tr_idx)}, val={len(vl_idx)}) ──")

            tr_f, tr_l = TV_FEATS[tr_idx], TV_LABELS[tr_idx]  # 512-dim raw
            vl_f, vl_l = TV_FEATS[vl_idx], TV_LABELS[vl_idx]

            # Build fresh model for this fold
            mod = _load_model(folder)
            qm  = mod.build_model(n_features=N_QUBITS, n_qubits=N_QUBITS,
                                   n_layers=N_LAYERS, task=TASK)
            net = ThinQuantumNet(qm)

            # Use PCA features (N_QUBITS dim) for quantum input
            tr_pca = TV_FEATS_PCA[tr_idx]
            vl_pca = TV_FEATS_PCA[vl_idx]

            # Override ThinQuantumNet.reducer with identity (PCA already reduced)
            net.reducer = nn.Linear(N_QUBITS, N_QUBITS)
            nn.init.eye_(net.reducer.weight)
            nn.init.zeros_(net.reducer.bias)

            _, fold_preds = _train_gradient_fold(
                net, tr_pca, tr_l, vl_pca, vl_l, EPOCHS_PER_FOLD,
                desc=f"{name} fold {fold}")

            oof_preds[vl_idx] = fold_preds
            sc = _score(vl_l, fold_preds)
            fold_scores.append(sc)
            metric_name = "MAE(g/dL)" if TASK=="regression" else "AUC"
            fold_val = _reg_metrics(vl_l,fold_preds)[0] if TASK=="regression" \
                       else _cls_metrics(vl_l.astype(int), fold_preds)[2]
            print(f"     Fold {fold} {metric_name} = {fold_val:.4f}")

        cv_mean = np.mean(fold_scores)
        cv_std  = np.std(fold_scores)
        print(f"  CV {metric_name}: {cv_mean:.4f} ± {cv_std:.4f}")

        # ── Final retrain on full Train+Val ──────────────────
        print(f"  Retraining on full Train+Val ({len(TV_FEATS_PCA)} samples)...")
        mod  = _load_model(folder)
        qm   = mod.build_model(n_features=N_QUBITS, n_qubits=N_QUBITS,
                                n_layers=N_LAYERS, task=TASK)
        net  = ThinQuantumNet(qm)
        net.reducer = nn.Linear(N_QUBITS, N_QUBITS)
        nn.init.eye_(net.reducer.weight); nn.init.zeros_(net.reducer.bias)
        net, _ = _train_gradient_fold(
            net, TV_FEATS_PCA, TV_LABELS,
            TV_FEATS_PCA[:max(4,BATCH_SIZE)], TV_LABELS[:max(4,BATCH_SIZE)],
            EPOCHS_FINAL)

        # ── Predict on test set ──────────────────────────────
        test_preds = _predict_gradient(net, TE_FEATS_PCA)

        OOF_PREDS[name]  = oof_preds
        TEST_PREDS[name] = test_preds
        CV_SCORES[name]  = cv_mean

        elapsed = round(time.time()-t0, 1)
        if TASK=="regression":
            te_mae,te_rmse = _reg_metrics(TE_LABELS, test_preds)
            print(f"  ✅ Test  MAE={te_mae:.3f} g/dL  RMSE={te_rmse:.3f}  "
                  f"CV={cv_mean:.4f}±{cv_std:.4f}  ({elapsed}s)")
            MODEL_RESULTS.append({"name":name,"type":"gradient","status":"ok",
                "cv_score":round(cv_mean,4),"cv_std":round(cv_std,4),
                "test_mae":round(te_mae,3),"test_rmse":round(te_rmse,3),
                "time_s":elapsed})
        else:
            te_acc,te_f1,te_auc = _cls_metrics(TE_LABELS.astype(int), test_preds)
            print(f"  ✅ Test  acc={te_acc:.3f}  f1={te_f1:.3f}  auc={te_auc:.3f}  "
                  f"CV={cv_mean:.4f}±{cv_std:.4f}  ({elapsed}s)")
            MODEL_RESULTS.append({"name":name,"type":"gradient","status":"ok",
                "cv_score":round(cv_mean,4),"cv_std":round(cv_std,4),
                "test_acc":round(te_acc,3),"test_f1":round(te_f1,3),
                "test_auc":round(te_auc,3),"time_s":elapsed})

    except Exception as e:
        print(f"  ❌ FAILED: {e}"); traceback.print_exc()
        OOF_PREDS[name]  = np.full(len(TV_LABELS), np.nan)
        TEST_PREDS[name] = np.full(len(TE_LABELS), np.nan)
        CV_SCORES[name]  = np.nan
        MODEL_RESULTS.append({"name":name,"type":"gradient",
                               "status":"failed","error":str(e)[:100]})


def run_kfold_svm(name, folder):
    """K-fold CV for SVM/kernel-based quantum models (fit/predict interface)."""
    print(f"\n{'='*62}")
    print(f"  [SVM] {name}")
    print(f"{'='*62}")
    t0 = time.time()
    kfold, strat_labels = _get_kfold()

    oof_preds   = np.zeros(len(TV_FEATS_PCA))
    fold_scores = []

    # For classification convert normalised labels to binary
    tv_labels_svm = (TV_LABELS>=0.5).astype(int) if TASK=="classification" \
                    else TV_LABELS
    te_labels_svm = (TE_LABELS>=0.5).astype(int) if TASK=="classification" \
                    else TE_LABELS

    try:
        _add(folder)
        for fold, (tr_idx, vl_idx) in enumerate(
                kfold.split(TV_FEATS_PCA, strat_labels), 1):

            print(f"  ── Fold {fold}/{K_FOLDS}  "
                  f"(train={len(tr_idx)}, val={len(vl_idx)}) ──")

            mod = _load_model(folder)
            _, est = mod.build_model(n_qubits=N_QUBITS, task=TASK)

            est.fit(TV_FEATS_PCA[tr_idx], tv_labels_svm[tr_idx])

            if TASK=="classification" and hasattr(est,"predict_proba"):
                p = est.predict_proba(TV_FEATS_PCA[vl_idx])
                fold_preds = p[:,1] if p.ndim==2 else p
            else:
                fold_preds = est.predict(TV_FEATS_PCA[vl_idx]).astype(float)

            oof_preds[vl_idx] = fold_preds
            sc  = _score(tv_labels_svm[vl_idx], fold_preds)
            fold_scores.append(sc)
            metric_name = "MAE(g/dL)" if TASK=="regression" else "AUC"
            fv = _reg_metrics(TV_LABELS[vl_idx],fold_preds)[0] if TASK=="regression" \
                 else _cls_metrics(tv_labels_svm[vl_idx],fold_preds)[2]
            print(f"     Fold {fold} {metric_name} = {fv:.4f}")

        cv_mean = np.mean(fold_scores); cv_std = np.std(fold_scores)
        print(f"  CV {metric_name}: {cv_mean:.4f} ± {cv_std:.4f}")

        # ── Final retrain on full Train+Val ──────────────────
        print(f"  Retraining on full Train+Val...")
        mod  = _load_model(folder)
        _, est = mod.build_model(n_qubits=N_QUBITS, task=TASK)
        est.fit(TV_FEATS_PCA, tv_labels_svm)

        if TASK=="classification" and hasattr(est,"predict_proba"):
            tp = est.predict_proba(TE_FEATS_PCA)
            test_preds = tp[:,1] if tp.ndim==2 else tp
        else:
            test_preds = est.predict(TE_FEATS_PCA).astype(float)

        OOF_PREDS[name]  = oof_preds
        TEST_PREDS[name] = test_preds
        CV_SCORES[name]  = cv_mean

        elapsed = round(time.time()-t0, 1)
        if TASK=="regression":
            te_mae,te_rmse = _reg_metrics(TE_LABELS, test_preds)
            print(f"  ✅ Test  MAE={te_mae:.3f} g/dL  RMSE={te_rmse:.3f}  "
                  f"CV={cv_mean:.4f}±{cv_std:.4f}  ({elapsed}s)")
            MODEL_RESULTS.append({"name":name,"type":"svm","status":"ok",
                "cv_score":round(cv_mean,4),"cv_std":round(cv_std,4),
                "test_mae":round(te_mae,3),"test_rmse":round(te_rmse,3),
                "time_s":elapsed})
        else:
            te_acc,te_f1,te_auc = _cls_metrics(te_labels_svm, test_preds)
            print(f"  ✅ Test  acc={te_acc:.3f}  f1={te_f1:.3f}  auc={te_auc:.3f}  "
                  f"CV={cv_mean:.4f}±{cv_std:.4f}  ({elapsed}s)")
            MODEL_RESULTS.append({"name":name,"type":"svm","status":"ok",
                "cv_score":round(cv_mean,4),"cv_std":round(cv_std,4),
                "test_acc":round(te_acc,3),"test_f1":round(te_f1,3),
                "test_auc":round(te_auc,3),"time_s":elapsed})

    except Exception as e:
        print(f"  ❌ FAILED: {e}"); traceback.print_exc()
        OOF_PREDS[name]  = np.full(len(TV_LABELS), np.nan)
        TEST_PREDS[name] = np.full(len(TE_LABELS), np.nan)
        CV_SCORES[name]  = np.nan
        MODEL_RESULTS.append({"name":name,"type":"svm",
                               "status":"failed","error":str(e)[:100]})

print("K-fold runners ready. ✅")


---
## Section 6 — Run All Models (K-Fold)

### QSVM
*QSVM — Nature 567 (2019)*

In [ ]:
if RUN["QSVM"]:
    run_kfold_svm("QSVM", "01_QSVM_QuantumEnhancedFeatureSpaces")
else:
    print("QSVM skipped.")


### VQC
*VQC — arXiv 1802.06002 (2018)*

In [ ]:
if RUN["VQC"]:
    run_kfold_gradient("VQC", "02_VQC_VariationalQuantumClassifier")
else:
    print("VQC skipped.")


### QCNN
*QCNN — Nature Physics 15 (2019)*

In [ ]:
if RUN["QCNN"]:
    run_kfold_gradient("QCNN", "03_QCNN_QuantumConvolutionalNeuralNet")
else:
    print("QCNN skipped.")


### TorchQuantum_VQA
*TorchQuantum VQA — Nat. Rev. Physics (2021)*

In [ ]:
if RUN["TorchQuantum_VQA"]:
    run_kfold_gradient("TorchQuantum VQA", "04_VQA_TorchQuantum_Framework")
else:
    print("TorchQuantum VQA skipped.")


### DataReUploading
*Data Re-Uploading — Quantum 4,226 (2020)*

In [ ]:
if RUN["DataReUploading"]:
    run_kfold_gradient("Data Re-Uploading", "05_DataReUploading_UniversalClassifier")
else:
    print("Data Re-Uploading skipped.")


### QuantumTransferLearning
*Quantum Transfer Learning — Quantum 4,340 (2020)*

In [ ]:
if RUN["QuantumTransferLearning"]:
    run_kfold_gradient("Quantum Transfer Learning", "06_QuantumTransferLearning")
else:
    print("Quantum Transfer Learning skipped.")


### QuantumKernel
*Quantum Kernel — PRX Quantum (2021)*

In [ ]:
if RUN["QuantumKernel"]:
    run_kfold_svm("Quantum Kernel", "07_QuantumKernelMethods")
else:
    print("Quantum Kernel skipped.")


### QuantumRandForest
*Quantum Random Forest — Springer QMI (2023)*

In [ ]:
if RUN["QuantumRandForest"]:
    run_kfold_svm("Quantum Random Forest", "08_QuantumRandomForest")
else:
    print("Quantum Random Forest skipped.")


### QBoost
*QBoost — ACML 2012*

In [ ]:
if RUN["QBoost"]:
    run_kfold_svm("QBoost", "09_QBoost_QuantumBoosting")
else:
    print("QBoost skipped.")


### CVQNN
*CV-QNN — Phys Rev Research (2019)*

In [ ]:
if RUN["CVQNN"]:
    run_kfold_gradient("CV-QNN", "10_CVQNN_ContinuousVariableQNN")
else:
    print("CV-QNN skipped.")


### QSVM_Medical
*BONUS — QSVM Medical*

In [ ]:
if RUN["QSVM_Medical"]:
    run_kfold_svm("BONUS", "11_BONUS_QSVM_MedicalClassifier")
else:
    print("BONUS skipped.")


### QCNN_Phases
*BONUS — QCNN Phases (Cong et al.)*

In [ ]:
if RUN["QCNN_Phases"]:
    run_kfold_gradient("BONUS", "12_BONUS_QCNN_PhasesOfMatter")
else:
    print("BONUS skipped.")


### MetaLearning_QNN
*BONUS — Meta-Learning QNN*

In [ ]:
if RUN["MetaLearning_QNN"]:
    run_kfold_gradient("BONUS", "13_BONUS_VQA_MetaLearning_QNN")
else:
    print("BONUS skipped.")


---
## Section 7 — Build Ensemble & Final Test Evaluation

Three ensemble strategies applied to the collected OOF predictions:
1. **Simple Average** — equal weight to all models
2. **Weighted Average** — weight ∝ CV score (better model → higher weight)
3. **Stacking** — Ridge (regression) or LogisticRegression (classification)
   meta-learner trained on OOF predictions, predicts on test

The **held-out test set is only evaluated here** — it was never used during K-fold.


In [ ]:
import warnings, traceback, copy
import numpy as np
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
import torch, torch.nn as nn
warnings.filterwarnings("ignore")

# ── Build input matrices ──────────────────────────────────────
valid_names = [n for n, p in OOF_PREDS.items()
               if not np.isnan(p).any() and not np.isnan(CV_SCORES.get(n, np.nan))]
skipped     = [n for n in OOF_PREDS if n not in valid_names]

print("=" * 72)
print(f"  ENSEMBLE ENGINE  —  Task: {TASK.upper()}  |  K={K_FOLDS} folds")
print("=" * 72)
print(f"  Valid models  : {len(valid_names)}  → {valid_names}")
if skipped:
    print(f"  Skipped/NaN   : {skipped}")
print("-" * 72)

if len(valid_names) == 0:
    print("  ✗  No valid models — cannot build ensemble.")
else:
    OOF_mat  = np.stack([OOF_PREDS[n]  for n in valid_names], axis=1)  # (Ntv, M)
    TEST_mat = np.stack([TEST_PREDS[n] for n in valid_names], axis=1)  # (Nte, M)
    tv_y = (TV_LABELS >= 0.5).astype(int) if TASK == "classification" else TV_LABELS
    te_y = (TE_LABELS >= 0.5).astype(int) if TASK == "classification" else TE_LABELS
    cv_scores_arr = np.array([CV_SCORES[n] for n in valid_names])

    ensemble_results = {}

    def _eval_ensemble(key, label, oof_pred, test_pred):
        oof_sc = _score(tv_y, oof_pred)
        if TASK == "regression":
            te_mae, te_rmse = _reg_metrics(TE_LABELS, test_pred)
            ensemble_results[key] = {
                "label": label, "oof_score": oof_sc,
                "test_mae": te_mae, "test_rmse": te_rmse,
            }
            print(f"  [{label:<30s}]  OOF MAE={oof_sc:.4f}  "
                  f"Test MAE={te_mae:.3f} g/dL  RMSE={te_rmse:.3f}")
        else:
            te_acc, te_f1, te_auc = _cls_metrics(te_y, test_pred)
            ensemble_results[key] = {
                "label": label, "oof_score": oof_sc,
                "test_acc": te_acc, "test_f1": te_f1, "test_auc": te_auc,
            }
            print(f"  [{label:<30s}]  OOF AUC={oof_sc:.4f}  "
                  f"Acc={te_acc:.3f}  F1={te_f1:.3f}  AUC={te_auc:.3f}")

    # ── 1. Simple Average ─────────────────────────────────────
    if RUN_ENSEMBLE.get("simple_avg"):
        _eval_ensemble("simple_avg", "Simple Average",
                       OOF_mat.mean(axis=1), TEST_mat.mean(axis=1))

    # ── 2. Weighted Average ───────────────────────────────────
    if RUN_ENSEMBLE.get("weighted_avg"):
        w = (1.0 / (cv_scores_arr + 1e-9)) if TASK == "regression" else cv_scores_arr
        w = w / w.sum()
        _eval_ensemble("weighted_avg", "Weighted Average",
                       (OOF_mat * w).sum(axis=1), (TEST_mat * w).sum(axis=1))

    # ── 3. Rank Average ───────────────────────────────────────
    if RUN_ENSEMBLE.get("rank_avg"):
        from scipy.stats import rankdata
        oof_ranks  = np.stack([rankdata(OOF_mat[:,i])  for i in range(OOF_mat.shape[1])],  axis=1)
        test_ranks = np.stack([rankdata(TEST_mat[:,i]) for i in range(TEST_mat.shape[1])], axis=1)
        oof_ra  = (oof_ranks.mean(axis=1)  - 1) / max(oof_ranks.shape[0]  - 1, 1)
        test_ra = (test_ranks.mean(axis=1) - 1) / max(test_ranks.shape[0] - 1, 1)
        _eval_ensemble("rank_avg", "Rank Average", oof_ra, test_ra)

    # ── 4. Stacking — Ridge / LogReg ──────────────────────────
    if RUN_ENSEMBLE.get("stacking_ridge"):
        try:
            if TASK == "regression":
                meta = Ridge(alpha=1.0)
                meta.fit(OOF_mat, tv_y)
                _eval_ensemble("stacking_ridge", "Stacking (Ridge)",
                               meta.predict(OOF_mat), meta.predict(TEST_mat))
            else:
                meta = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
                meta.fit(OOF_mat, tv_y)
                _eval_ensemble("stacking_ridge", "Stacking (LogReg)",
                               meta.predict_proba(OOF_mat)[:, 1],
                               meta.predict_proba(TEST_mat)[:, 1])
        except Exception as e:
            print(f"  [Stacking Ridge] FAILED — {e}")

    # ── 5. Stacking — GBM ─────────────────────────────────────
    if RUN_ENSEMBLE.get("stacking_gbm"):
        try:
            if TASK == "regression":
                meta = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED)
                meta.fit(OOF_mat, tv_y)
                _eval_ensemble("stacking_gbm", "Stacking (GBM)",
                               meta.predict(OOF_mat), meta.predict(TEST_mat))
            else:
                meta = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED)
                meta.fit(OOF_mat, tv_y)
                _eval_ensemble("stacking_gbm", "Stacking (GBM)",
                               meta.predict_proba(OOF_mat)[:, 1],
                               meta.predict_proba(TEST_mat)[:, 1])
        except Exception as e:
            print(f"  [Stacking GBM] FAILED — {e}")

    # ── 6. Voting / Median ────────────────────────────────────
    if RUN_ENSEMBLE.get("voting"):
        if TASK == "regression":
            _eval_ensemble("voting", "Median Vote (Reg)",
                           np.median(OOF_mat, axis=1), np.median(TEST_mat, axis=1))
        else:
            thresh = CLS_CONFIG["threshold"]
            oof_v  = (OOF_mat  >= thresh).astype(int).mean(axis=1)
            test_v = (TEST_mat >= thresh).astype(int).mean(axis=1)
            _eval_ensemble("voting", "Majority Vote (Cls)", oof_v, test_v)

    # ── 7. Best Single (reference) ────────────────────────────
    if RUN_ENSEMBLE.get("best_single"):
        best_idx = int(np.argmin(cv_scores_arr)) if TASK=="regression"                    else int(np.argmax(cv_scores_arr))
        best_name = valid_names[best_idx]
        _eval_ensemble("best_single", f"Best Single ({best_name[:12]})",
                       OOF_mat[:, best_idx], TEST_mat[:, best_idx])

    # ── 8. Top-K Average (NEW) ────────────────────────────────
    if RUN_ENSEMBLE.get("top_k") and len(valid_names) >= 2:
        k = min(TOP_K, len(valid_names))
        if TASK == "regression":
            top_idx = np.argsort(cv_scores_arr)[:k]       # lowest MAE
        else:
            top_idx = np.argsort(cv_scores_arr)[-k:]      # highest AUC
        _eval_ensemble("top_k", f"Top-{k} Average",
                       OOF_mat[:, top_idx].mean(axis=1),
                       TEST_mat[:, top_idx].mean(axis=1))
        print(f"    Top-{k} models: {[valid_names[i] for i in top_idx]}")

    # ── 9. Trimmed Mean (NEW) — removes outlier predictions ───
    if RUN_ENSEMBLE.get("trimmed_mean") and OOF_mat.shape[1] >= 3:
        from scipy.stats import trim_mean
        _trim = 1.0 / OOF_mat.shape[1]   # trim ~1 model from each side
        oof_tm  = np.array([trim_mean(OOF_mat[i,:],  _trim) for i in range(len(tv_y))])
        test_tm = np.array([trim_mean(TEST_mat[i,:], _trim) for i in range(len(te_y))])
        _eval_ensemble("trimmed_mean", "Trimmed Mean",  oof_tm, test_tm)

    # ── 10. Neural Stacker MLP (NEW) ─────────────────────────
    if RUN_ENSEMBLE.get("neural_stacker") and OOF_mat.shape[1] >= 2:
        try:
            import torch
            import torch.nn as nn
            from torch.utils.data import TensorDataset, DataLoader as TDL

            M_in = OOF_mat.shape[1]
            X_tv_t = torch.tensor(OOF_mat,  dtype=torch.float32)
            X_te_t = torch.tensor(TEST_mat, dtype=torch.float32)
            y_tv_t = torch.tensor(tv_y,     dtype=torch.float32)

            class MetaMLP(nn.Module):
                def __init__(self, m_in):
                    super().__init__()
                    self.net = nn.Sequential(
                        nn.Linear(m_in, 16), nn.ReLU(), nn.Dropout(0.3),
                        nn.Linear(16, 8),    nn.ReLU(),
                        nn.Linear(8, 1),     nn.Sigmoid() if TASK=="classification" else nn.Identity()
                    )
                def forward(self, x): return self.net(x).squeeze(-1)

            meta_net = MetaMLP(M_in)
            opt_m = torch.optim.Adam(meta_net.parameters(), lr=5e-3, weight_decay=1e-4)
            loss_m = nn.MSELoss() if TASK=="regression" else nn.BCELoss()
            ds = TensorDataset(X_tv_t, y_tv_t)
            dl = TDL(ds, batch_size=min(32, len(ds)), shuffle=True)

            for ep in range(80):
                meta_net.train()
                for xb, yb in dl:
                    opt_m.zero_grad()
                    loss_m(meta_net(xb), yb).backward()
                    opt_m.step()

            meta_net.eval()
            with torch.no_grad():
                oof_mlp  = meta_net(X_tv_t).numpy()
                test_mlp = meta_net(X_te_t).numpy()
            _eval_ensemble("neural_stacker", "Neural Stacker (MLP)", oof_mlp, test_mlp)
        except Exception as e:
            print(f"  [Neural Stacker] FAILED — {e}")

    # ── 11. Blending (NEW) ────────────────────────────────────
    if RUN_ENSEMBLE.get("blending") and len(valid_names) >= 2:
        try:
            np.random.seed(SEED)
            blend_size = max(4, int(0.15 * len(tv_y)))
            blend_idx  = np.random.choice(len(tv_y), blend_size, replace=False)
            train_idx  = np.setdiff1d(np.arange(len(tv_y)), blend_idx)

            # Fit meta-learner on the blend-val portion (TV) only
            if TASK == "regression":
                meta_b = Ridge(alpha=0.5)
            else:
                meta_b = LogisticRegression(C=2.0, max_iter=300, random_state=SEED)
            meta_b.fit(OOF_mat[blend_idx], tv_y[blend_idx])

            oof_blend  = meta_b.predict(OOF_mat)   if TASK=="regression"                          else meta_b.predict_proba(OOF_mat)[:,1]
            test_blend = meta_b.predict(TEST_mat)  if TASK=="regression"                          else meta_b.predict_proba(TEST_mat)[:,1]
            if TASK == "regression":
                test_blend = np.clip(test_blend, 0.0, 1.0)
            _eval_ensemble("blending", "Blending (Ridge/LogReg)", oof_blend, test_blend)
        except Exception as e:
            print(f"  [Blending] FAILED — {e}")

    # ── 12. Bayesian Averaging (NEW) ─────────────────────────
    if RUN_ENSEMBLE.get("bayesian_avg"):
        try:
            T = 0.5   # temperature: lower = sharper selection
            if TASK == "regression":
                # lower MAE → higher weight → negate scores
                log_w = -cv_scores_arr / T
            else:
                log_w =  cv_scores_arr / T
            log_w -= log_w.max()   # numerical stability
            w_bay = np.exp(log_w)
            w_bay /= w_bay.sum()
            _eval_ensemble("bayesian_avg", "Bayesian Averaging",
                           (OOF_mat  * w_bay).sum(axis=1),
                           (TEST_mat * w_bay).sum(axis=1))
            top3 = np.argsort(w_bay)[::-1][:3]
            print(f"    Bayesian top weights: "
                  + ", ".join(f"{valid_names[i][:14]}={w_bay[i]:.3f}" for i in top3))
        except Exception as e:
            print(f"  [Bayesian Avg] FAILED — {e}")

    print("-" * 72)
    print(f"  Ensembles completed: {len(ensemble_results)}")


---
## Section 8 — Full Results Summary & Plots

In [ ]:
# ════════════════════════════════════════════════════════════════
#   FINAL TEXT SUMMARY
# ════════════════════════════════════════════════════════════════
import textwrap, math, pandas as pd

SEP = "═" * 74
sep = "─" * 74

def _banner(title):
    pad = (74 - len(title) - 2) // 2
    print(f"\n{SEP}")
    print(f"{'═'*pad} {title} {'═'*(74 - pad - len(title) - 2)}")
    print(SEP)

def _medal(rank):
    return ["🥇","🥈","🥉"][rank] if rank < 3 else f"  {rank+1}."

# ── A. Per-model K-fold results ────────────────────────────────
if MODEL_RESULTS:
    _banner(f"PER-MODEL {K_FOLDS}-FOLD RESULTS  ·  {TASK.upper()}")
    df_m = pd.DataFrame(MODEL_RESULTS)
    metric_col = "test_mae" if TASK=="regression" else "test_auc"
    sort_asc   = TASK == "regression"
    ok_rows  = df_m[df_m["status"]=="ok"].sort_values(metric_col, ascending=sort_asc)
    bad_rows = df_m[df_m["status"]!="ok"]

    if TASK == "regression":
        hdr = f"  {'Rank':<5} {'Model':<28} {'Type':<9} {'CV MAE':>9} {'±':>5} {'TestMAE':>9} {'RMSE':>7}  {'Time':>6}"
    else:
        hdr = f"  {'Rank':<5} {'Model':<28} {'Type':<9} {'CV AUC':>9} {'±':>5} {'Acc':>7} {'F1':>7} {'AUC':>7}  {'Time':>6}"
    print(hdr); print(f"  {sep}")

    for rank, (_, row) in enumerate(ok_rows.iterrows()):
        cv  = f"{row.get('cv_score', float('nan')):.4f}"
        std = f"{row.get('cv_std', 0):.4f}"
        if TASK == "regression":
            metrics = (f"{row.get('test_mae', float('nan')):>9.3f}"
                       f"{row.get('test_rmse', float('nan')):>8.3f}")
        else:
            metrics = (f"{row.get('test_acc', float('nan')):>7.3f}"
                       f"{row.get('test_f1',  float('nan')):>8.3f}"
                       f"{row.get('test_auc', float('nan')):>8.3f}")
        t = row.get("time_s", 0)
        ts = f"{t/60:.1f}m" if t >= 60 else f"{t:.0f}s"
        print(f"  {_medal(rank):<5} {row['name']:<28} {row.get('type','?'):<9} "
              f"{cv:>9}{std:>5}{metrics}  {ts:>6}")
    if not bad_rows.empty:
        print(f"\n  Failed:")
        for _, row in bad_rows.iterrows():
            print(f"    ✗  {row['name']}")

# ── B. Ensemble leaderboard ────────────────────────────────────
if "ensemble_results" in dir() and ensemble_results:
    _banner("ENSEMBLE LEADERBOARD  ·  All 12 Methods")
    metric_col = "test_mae" if TASK=="regression" else "test_auc"
    rows = sorted(ensemble_results.values(),
                  key=lambda r: r.get(metric_col, 9999 if TASK=="regression" else 0),
                  reverse=TASK!="regression")
    if TASK == "regression":
        print(f"  {'Rank':<5} {'Ensemble':<32} {'OOF MAE':>9} {'Test MAE':>10} {'RMSE':>8}")
        print(f"  {sep}")
        for rank, r in enumerate(rows):
            flag = "  ← BEST" if rank == 0 else ""
            print(f"  {_medal(rank):<5} {r['label']:<32}"
                  f"{r['oof_score']:>9.4f}"
                  f"{r.get('test_mae', float('nan')):>10.3f}"
                  f"{r.get('test_rmse', float('nan')):>8.3f}{flag}")
    else:
        print(f"  {'Rank':<5} {'Ensemble':<32} {'OOF AUC':>9} {'Acc':>8} {'F1':>8} {'AUC':>8}")
        print(f"  {sep}")
        for rank, r in enumerate(rows):
            flag = "  ← BEST" if rank == 0 else ""
            print(f"  {_medal(rank):<5} {r['label']:<32}"
                  f"{r['oof_score']:>9.4f}"
                  f"{r.get('test_acc', float('nan')):>8.3f}"
                  f"{r.get('test_f1',  float('nan')):>8.3f}"
                  f"{r.get('test_auc', float('nan')):>8.3f}{flag}")

    # ── C. Best ensemble vs best single ────────────────────────
    _banner("ENSEMBLE vs BEST SINGLE MODEL")
    if MODEL_RESULTS:
        df_ok2 = pd.DataFrame([r for r in MODEL_RESULTS if r.get("status")=="ok"])
        if not df_ok2.empty:
            metric_col = "test_mae" if TASK=="regression" else "test_auc"
            best_m = df_ok2.loc[df_ok2[metric_col].idxmin() if TASK=="regression"
                                 else df_ok2[metric_col].idxmax()]
            best_e = rows[0]
            m_val  = best_m[metric_col]
            e_val  = best_e.get(metric_col, float("nan"))
            gain   = ((m_val - e_val) / m_val * 100) if TASK=="regression" \
                     else ((e_val - m_val) / max(m_val, 1e-9) * 100)
            print(f"\n  Best single model : {best_m['name']:<28}  {metric_col}={m_val:.3f}")
            print(f"  Best ensemble     : {best_e['label']:<28}  {metric_col}={e_val:.3f}")
            if gain > 0:
                print(f"  → Ensemble improves by {gain:.1f}%  🎉")
            else:
                print(f"  → Single model is better by {-gain:.1f}%")

    # ── D. Model diversity ────────────────────────────────────
    valid_oof = [n for n, p in OOF_PREDS.items() if not np.isnan(p).any()]
    if len(valid_oof) > 1:
        _banner("MODEL DIVERSITY  (OOF Prediction Correlations)")
        oof_df   = pd.DataFrame({n: OOF_PREDS[n] for n in valid_oof})
        corr_mat = oof_df.corr()
        upper    = corr_mat.where(
            __import__("numpy").triu(__import__("numpy").ones(corr_mat.shape), k=1).astype(bool))
        vals = upper.stack().values
        print(f"\n  Mean pairwise Pearson r : {vals.mean():.3f}")
        print(f"  Min / Max correlation   : {vals.min():.3f} / {vals.max():.3f}")
        print(f"  (Ideal mean r < 0.85 for diverse ensemble)")

print(f"\n{SEP}")
print(f"  ✅  Complete.  {len(MODEL_RESULTS)} models  ·  "
      f"{len(ensemble_results) if 'ensemble_results' in dir() else 0} ensembles")
print(SEP)


---
## Section 8b — Comprehensive Visualization Dashboard

In [ ]:
# ════════════════════════════════════════════════════════════════
#   COMPREHENSIVE ENSEMBLE VISUALIZATION DASHBOARD
# ════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")

ok_model_rows = [r for r in MODEL_RESULTS if r.get("status") == "ok"]
has_ensemble  = "ensemble_results" in dir() and ensemble_results

metric     = "test_mae"  if TASK == "regression" else "test_auc"
xlabel     = "Test MAE (g/dL)" if TASK == "regression" else "Test AUC"
ascending  = TASK == "regression"

if ok_model_rows or has_ensemble:
    fig = plt.figure(figsize=(24, 22))
    gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.38)

    colors_plasma = lambda n: plt.cm.plasma(np.linspace(0.15, 0.85, n))
    colors_viridis= lambda n: plt.cm.viridis(np.linspace(0.15, 0.85, n))

    # ── Panel 1: Per-model test metric (top-left wide) ─────────
    ax1 = fig.add_subplot(gs[0, :2])
    if ok_model_rows:
        df_ok = pd.DataFrame(ok_model_rows).sort_values(metric, ascending=ascending)
        bars = ax1.barh(df_ok["name"], df_ok[metric],
                        color=colors_plasma(len(df_ok)), edgecolor="white", linewidth=0.5)
        bars[0 if ascending else -1].set_edgecolor("gold")
        bars[0 if ascending else -1].set_linewidth(2.5)
        for bar, val in zip(bars, df_ok[metric]):
            ax1.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                     f"{val:.3f}", va="center", fontsize=8, fontweight="bold")
        ax1.set_xlabel(xlabel, fontsize=10)
        ax1.set_title(f"Per-Model  —  {xlabel}  (sorted best→worst)",
                      fontsize=10, fontweight="bold")
        ax1.grid(axis="x", alpha=0.3, linestyle="--")
    ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

    # ── Panel 2: CV vs Test scatter (top-right) ────────────────
    ax2 = fig.add_subplot(gs[0, 2])
    if ok_model_rows:
        df_ok2 = pd.DataFrame(ok_model_rows)
        if "cv_score" in df_ok2.columns and metric in df_ok2.columns:
            ax2.scatter(df_ok2["cv_score"], df_ok2[metric],
                        s=90, c=colors_plasma(len(df_ok2)),
                        edgecolors="k", linewidths=0.4, zorder=3)
            for _, row in df_ok2.iterrows():
                ax2.annotate(row["name"][:10], (row["cv_score"], row[metric]),
                             fontsize=6, xytext=(3,2), textcoords="offset points")
            # diagonal reference
            lo = min(df_ok2["cv_score"].min(), df_ok2[metric].min()) * 0.97
            hi = max(df_ok2["cv_score"].max(), df_ok2[metric].max()) * 1.03
            ax2.plot([lo,hi],[lo,hi],"k--",alpha=0.35,linewidth=1)
        ax2.set_xlabel("CV Score (K-Fold)", fontsize=9)
        ax2.set_ylabel(f"Test {xlabel}", fontsize=9)
        ax2.set_title("CV Score vs Test
(ideal: on diagonal)", fontsize=9, fontweight="bold")
        ax2.grid(True, alpha=0.3, linestyle="--")
    ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

    # ── Panel 3: Ensemble leaderboard (middle-left wide) ───────
    ax3 = fig.add_subplot(gs[1, :2])
    if has_ensemble:
        ens_rows = sorted(ensemble_results.values(),
                          key=lambda r: r.get(metric, 9999 if ascending else 0),
                          reverse=not ascending)
        ens_labels = [r["label"] for r in ens_rows]
        ens_vals   = [r.get(metric, float("nan")) for r in ens_rows]
        ens_colors = colors_viridis(len(ens_rows))
        bars3 = ax3.barh(ens_labels, ens_vals, color=ens_colors,
                         edgecolor="white", linewidth=0.5)
        bars3[0].set_edgecolor("gold"); bars3[0].set_linewidth(2.5)
        for bar, val in zip(bars3, ens_vals):
            if not (val != val):
                ax3.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                         f"{val:.3f}", va="center", fontsize=8, fontweight="bold")
        ax3.set_xlabel(xlabel, fontsize=10)
        ax3.set_title(f"Ensemble Methods  —  {xlabel}  (sorted best→worst)",
                      fontsize=10, fontweight="bold")
        ax3.grid(axis="x", alpha=0.3, linestyle="--")
    ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)

    # ── Panel 4: OOF correlation heatmap (middle-right) ────────
    valid_oof = [n for n, p in OOF_PREDS.items() if not np.isnan(p).any()]
    ax4 = fig.add_subplot(gs[1, 2])
    if len(valid_oof) > 1:
        corr = pd.DataFrame({n: OOF_PREDS[n] for n in valid_oof}).corr()
        n    = len(valid_oof)
        im   = ax4.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
        ax4.set_xticks(range(n)); ax4.set_xticklabels([x[:7] for x in valid_oof],
                       rotation=45, ha="right", fontsize=5.5)
        ax4.set_yticks(range(n)); ax4.set_yticklabels([x[:7] for x in valid_oof], fontsize=5.5)
        for i in range(n):
            for j in range(n):
                ax4.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                         fontsize=4.5, color="k" if abs(corr.iloc[i,j])<0.7 else "w")
        plt.colorbar(im, ax=ax4, label="r", shrink=0.8)
        ax4.set_title("OOF Prediction
Correlation Heatmap
(low r = diverse)",
                      fontsize=8.5, fontweight="bold")

    # ── Panel 5: Ensemble OOF MAE/AUC comparison bar ───────────
    ax5 = fig.add_subplot(gs[2, :2])
    if has_ensemble and ok_model_rows:
        # combine best-single-model and all ensembles for comparison
        compare = []
        best_single_row = min(ok_model_rows, key=lambda r: r.get(metric, 9999))                           if ascending else max(ok_model_rows, key=lambda r: r.get(metric,0))
        compare.append({"label": f"★ Best Single
({best_single_row['name'][:15]})",
                        "val": best_single_row[metric], "oof": float("nan"), "type": "single"})
        for key, r in ensemble_results.items():
            if key != "best_single":
                compare.append({"label": r["label"], "val": r.get(metric, float("nan")),
                                 "oof": r.get("oof_score", float("nan")), "type": "ensemble"})
        compare.sort(key=lambda x: x["val"], reverse=not ascending)
        c_labels = [c["label"] for c in compare]
        c_test   = [c["val"]  for c in compare]
        c_oof    = [c["oof"]  for c in compare]
        x = np.arange(len(compare))
        w = 0.38
        bars5a = ax5.bar(x - w/2, c_test, w, label="Test", color="#4C72B0", alpha=0.85)
        valid_oof_vals = [(xi, v) for xi,(v,t) in enumerate(zip(c_oof, [c["type"] for c in compare])) if not (v!=v) and t=="ensemble"]
        if valid_oof_vals:
            xi_v, v_v = zip(*valid_oof_vals)
            ax5.bar([x[i]-w/2+w for i in xi_v], v_v, w*0.7, label="OOF", color="#DD8452", alpha=0.75)
        ax5.set_xticks(x); ax5.set_xticklabels(c_labels, rotation=30, ha="right", fontsize=7.5)
        ax5.set_ylabel(xlabel, fontsize=9)
        ax5.set_title("Ensemble vs Best-Single: Test & OOF Comparison", fontsize=10, fontweight="bold")
        ax5.legend(fontsize=8); ax5.grid(axis="y", alpha=0.3, linestyle="--")
    ax5.spines["top"].set_visible(False); ax5.spines["right"].set_visible(False)

    # ── Panel 6: Prediction distribution violin (bottom-right) ─
    ax6 = fig.add_subplot(gs[2, 2])
    if valid_oof:
        vdata  = [OOF_PREDS[n] for n in valid_oof]
        vnames = [n[:10] for n in valid_oof]
        parts = ax6.violinplot(vdata, showmedians=True, showextrema=True)
        for pc in parts["bodies"]: pc.set_alpha(0.60)
        ax6.set_xticks(range(1, len(vnames)+1))
        ax6.set_xticklabels(vnames, rotation=45, ha="right", fontsize=5.5)
        ax6.axhline(np.mean([np.median(d) for d in vdata]), color="red",
                    linestyle="--", alpha=0.5, linewidth=1, label="Grand median")
        ax6.set_ylabel("OOF Prediction (normalised Hb)", fontsize=8)
        ax6.set_title("OOF Prediction
Distribution per Model", fontsize=8.5, fontweight="bold")
        ax6.legend(fontsize=7); ax6.grid(axis="y", alpha=0.3, linestyle="--")
    ax6.spines["top"].set_visible(False); ax6.spines["right"].set_visible(False)

    # ── Panel 7: Ensemble weight heatmap ─────────────────────
    ax7 = fig.add_subplot(gs[3, :2])
    if has_ensemble:
        # Show which models have high weight in each ensemble method
        ens_keys = [k for k in RUN_ENSEMBLE if RUN_ENSEMBLE[k] and k in ensemble_results and k not in ("best_single",)]
        if len(ens_keys) > 0 and len(valid_names) > 0:
            weight_matrix = np.zeros((len(ens_keys), len(valid_names)))
            for ei, key in enumerate(ens_keys):
                if key == "simple_avg":
                    weight_matrix[ei] = 1.0 / len(valid_names)
                elif key == "weighted_avg":
                    w = (1/(cv_scores_arr+1e-9)) if TASK=="regression" else cv_scores_arr
                    weight_matrix[ei] = w / w.sum()
                elif key == "bayesian_avg":
                    T = 0.5
                    log_w = (-cv_scores_arr/T) if TASK=="regression" else (cv_scores_arr/T)
                    log_w -= log_w.max()
                    w_b = np.exp(log_w); weight_matrix[ei] = w_b / w_b.sum()
                elif key == "top_k":
                    k = min(TOP_K, len(valid_names))
                    top_idx = np.argsort(cv_scores_arr)[:k] if TASK=="regression"                               else np.argsort(cv_scores_arr)[-k:]
                    weight_matrix[ei, top_idx] = 1.0 / k
                else:
                    weight_matrix[ei] = 1.0 / len(valid_names)  # fallback

            im7 = ax7.imshow(weight_matrix, cmap="YlOrRd", aspect="auto", vmin=0)
            ax7.set_xticks(range(len(valid_names)))
            ax7.set_xticklabels([n[:12] for n in valid_names], rotation=35, ha="right", fontsize=6.5)
            ax7.set_yticks(range(len(ens_keys)))
            ax7.set_yticklabels([ensemble_results[k]["label"][:22] for k in ens_keys], fontsize=7)
            for ei in range(len(ens_keys)):
                for ni in range(len(valid_names)):
                    v = weight_matrix[ei, ni]
                    if v > 0.001:
                        ax7.text(ni, ei, f"{v:.2f}", ha="center", va="center",
                                 fontsize=5, color="k" if v < 0.4 else "w")
            plt.colorbar(im7, ax=ax7, label="Weight", shrink=0.8)
            ax7.set_title("Ensemble Model Weights Heatmap
(brighter = higher weight for that model)",
                          fontsize=9, fontweight="bold")

    # ── Panel 8: Residuals / Error analysis ──────────────────
    ax8 = fig.add_subplot(gs[3, 2])
    if has_ensemble and TASK == "regression":
        # Show absolute error distribution for best ensemble vs best single
        best_ens_key = min(ensemble_results, key=lambda k: ensemble_results[k].get("test_mae", 9999))
        best_ens_pred = TEST_PREDS.get(list(OOF_PREDS.keys())[0], None)

        if "best_single" in ensemble_results:
            best_single_name = [n for n in valid_names
                                if n == valid_names[int(np.argmin(cv_scores_arr))]
                               ][0] if valid_names else None
            if best_single_name and best_single_name in TEST_PREDS:
                bsp = TEST_PREDS[best_single_name]
                be_label = f"Best Single ({best_single_name[:12]})"
                bsp_mae  = ensemble_results["best_single"].get("test_mae", float("nan"))
            else:
                bsp = TEST_mat[:, 0]; be_label = "Best Single"; bsp_mae = float("nan")
        else:
            bsp = TEST_mat[:,0]; be_label = ""; bsp_mae = float("nan")

        ens_preds = (TEST_mat * (
            (1/cv_scores_arr) / (1/cv_scores_arr).sum())).sum(axis=1)  # weighted avg

        ens_err   = np.abs(_denorm_hb(ens_preds)    - _denorm_hb(TE_LABELS))
        sing_err  = np.abs(_denorm_hb(bsp)           - _denorm_hb(TE_LABELS))

        bins = np.linspace(0, max(ens_err.max(), sing_err.max()) * 1.05, 25)
        ax8.hist(sing_err, bins=bins, alpha=0.55, color="#4C72B0",
                 label=f"{be_label[:18]}
MAE={bsp_mae:.3f}", density=True)
        ax8.hist(ens_err, bins=bins, alpha=0.55, color="#DD8452",
                 label=f"Weighted Avg Ens
MAE={ensemble_results.get('weighted_avg',{}).get('test_mae', float('nan')):.3f}", density=True)
        ax8.axvline(np.mean(sing_err), color="#4C72B0", linestyle="--", linewidth=1.5)
        ax8.axvline(np.mean(ens_err),  color="#DD8452", linestyle="--", linewidth=1.5)
        ax8.set_xlabel("Absolute Error (g/dL)", fontsize=9)
        ax8.set_ylabel("Density", fontsize=9)
        ax8.set_title("Error Distribution
Best Single vs Ensemble", fontsize=9, fontweight="bold")
        ax8.legend(fontsize=6.5); ax8.grid(True, alpha=0.3, linestyle="--")
    elif has_ensemble and TASK == "classification":
        ax8.text(0.5, 0.5, "Error distribution\n(regression only)",
                 ha="center", va="center", transform=ax8.transAxes)
    ax8.spines["top"].set_visible(False); ax8.spines["right"].set_visible(False)

    fig.suptitle(
        f"QuantumHb — Ensemble Dashboard  |  {TASK.upper()}  |  K={K_FOLDS} folds  "
        f"|  {len(valid_names)} models  |  {len(ensemble_results)} ensembles",
        fontsize=14, fontweight="bold", y=1.005)
    plt.savefig(
        os.path.join(REPO_ROOT, "ensemble_results.png"), dpi=130, bbox_inches="tight")
    plt.show()
    print("\n  📊  Dashboard saved → ensemble_results.png")
